In [162]:
import polars as pl
import pandas as pd
import numpy as np
import altair as alt
import pyarrow
from numpy.ma.extras import unique
from polars import Null
import matplotlib.pyplot as plt
import squarify
import seaborn as sns

from func.UsefulFunctions import *

Find the missing people, so people who are on musicbrainz but not on wikidata

In [163]:
wikidataArtistDF = pd.read_parquet("./Data/Network_artistsWikidata.parquet")
mbArtistsData = pd.read_parquet("./Data/Network_personMBData.parquet")

display(wikidataArtistDF.head())
display(wikidataArtistDF[wikidataArtistDF["musicBrainzID"].isna()])
display(mbArtistsData.head())

,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
0,Q3176190,Q639669,Q6581072,musician,female,Q30,NaN,NaN,NaN,Q43343,https://en.wikipedia.org/wiki/Linda_Gail_Lewis,a2fe484f-4f35-4e0e-a954-2f6d5843b009,Q7544232,Smash Records,United States,folk music,Linda Gail Lewis
1,Q8065552,Q183945,Q6581097,record producer,male,Q145,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Zalon,19c43212-18cf-4f02-baff-b794de3cdf02,NaN,NaN,United Kingdom,NaN,Zalon
2,Q191819,Q806349,Q6581097,bandleader,male,Q174193,NaN,NaN,NaN,Q9730,https://en.wikipedia.org/wiki/George_Martin,26fa8b67-6c7f-406c-ad64-a1d070092df2,Q21077,Warner Music Group,United Kingdom of Great Britain and Ireland,classical music,George Martin
3,Q21934026,Q183945,Q6581097,record producer,male,NaN,NaN,NaN,NaN,Q37073,https://en.wikipedia.org/wiki/Andrew_Scheps,ee11737c-d50c-4b04-ae5f-56e1f6373116,NaN,NaN,NaN,pop music,Andrew Scheps
4,Q1005957,Q55960555,Q6581097,recording artist,male,Q28,NaN,NaN,NaN,NaN,NaN,6054baf8-3a47-4c3a-8c51-eb976f69ecfc,NaN,NaN,Hungary,NaN,László Földes


,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
8,Q100327193,Q1076502,Q6581097,choir director,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Kazimierz Danysz
10,Q60230254,Q488205,Q6581097,singer-songwriter,male,Q884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,South Korea,NaN,Sanha
13,Q2179893,Q36834,Q6581097,composer,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Johann Pavlis
14,Q135617645,Q158852,Q6581097,conductor,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lorenzo Barbirolli
17,Q111634846,Q36834,Q6581097,composer,male,Q145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,United Kingdom,NaN,George Alan Moorhouse
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1235898,Q130977202,Q158852,Q6581072,conductor,female,Q40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Austria,NaN,Katharina Wincor
1235900,Q115190045,Q36834,Q6581097,composer,male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Jaroslav Šimsa
1235902,Q114043574,Q177220,Q6581072,singer,female,Q215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Slovenia,NaN,Andreja Tomažič Hrvatin
1235903,Q12302475,Q639669,Q6581097,musician,male,Q43287,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Deutsches Kaiserreich,NaN,August Wilhelm Binneweitz


,id,name,type,gender,country,genres,isnis,ipis,life-span,genreList
0,fc926d23-4f61-4df5-8aaf-9f24109f660f,Zheng Yue Wen,Person,NaN,NaN,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
1,595058cd-36fe-4994-8933-df92ff68ebd7,DJ Seinfeld,Person,Male,ES,[{'id': '8ae6fbbe-2dd0-432e-a176-6678f48dffb4'...,[0000000465763535],[],"{'end': None, 'begin': None, 'ended': False}","[bassline, breakbeat, breaks, club, dance, dee..."
2,4c6b0be1-3e90-430b-b208-e87945144be0,Trishita Recs,Person,Female,IN,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
3,fe37b5c5-3c9f-48f4-9f9a-28939d0e3d77,Salvatore Marino,Person,Male,NaN,[],[],[],"{'end': None, 'begin': None, 'ended': False}",[]
4,c666baa0-ebbf-4103-8249-a53db676cece,Zoë Marie Federoff,Person,Female,US,[],[],[],"{'end': None, 'begin': '1992-05-25', 'ended': ...",[]


In [164]:
#Dataset of artists who are on MusicBrainz but not on Wikidata

#How many entities have a music brainz id
numberOfFolkWithMusicBrainzID = len(wikidataArtistDF[~wikidataArtistDF["musicBrainzID"].isnull()]["artist"].drop_duplicates())

print(f"There are {numberOfFolkWithMusicBrainzID} individuals with a music brainz id which is {numberOfFolkWithMusicBrainzID/len(wikidataArtistDF["artist"].unique())*100:.2f}% of the dataset.")

There are 179192 individuals with a music brainz id which is 40.28% of the dataset.


In [165]:
#Using ID to check the individuals
peopleNotOnWikidataByID = mbArtistsData[~mbArtistsData["id"].isin(wikidataArtistDF["musicBrainzID"].unique())]

print(f"There are {len(peopleNotOnWikidataByID["id"].unique())} people not on wikidata based on MusicBrainzID")

There are 1392611 people not on wikidata based on MusicBrainzID


In [166]:
#Using just names
peopleNotONWikidataByName = mbArtistsData[~mbArtistsData["name"].str.lower().isin(wikidataArtistDF["artistLabel"].str.lower().unique())]
print(f"There are {len(peopleNotONWikidataByName["id"].unique())} people not on wikidata based on name. Not a good metric to use")

There are 1325311 people not on wikidata based on name. Not a good metric to use


Look for the missing groups

In [167]:
wikidataGroupDF = pd.read_parquet("./Data/Network_groupWikidata.parquet")
mbGroupsData = pd.read_parquet("./Data/Network_groupMBData.parquet")


#WHO HAS A MUSICBRAINZ ID
groupsWithMBID = wikidataGroupDF[~wikidataGroupDF["musicBrainzID"].isnull()].drop_duplicates()
print(f"There are {len(groupsWithMBID["group"].unique())} groups with a music brainz id which is {len(groupsWithMBID["group"].unique())/len(wikidataGroupDF["group"].unique())*100:.2f}% of groups")
display(groupsWithMBID[groupsWithMBID["musicBrainzID"].str.contains("http")])
#This is how much of the musicBrainzDataset
group_withID = groupsWithMBID[["group", "groupLabel", "musicBrainzID"]].drop_duplicates()
display(group_withID[group_withID.duplicated(subset="group")])
print(len(group_withID[group_withID.duplicated(subset="group")]["group"].unique()))
groupsInMusicBrainzBasedOnID = wikidataGroupDF[wikidataGroupDF["musicBrainzID"].isin(mbGroupsData["id"].unique())]

print(f"There are {len(groupsInMusicBrainzBasedOnID["group"].unique())} in the musicBrainz dataset that are also on wikidata based on ids from wikidata")

There are 76257 groups with a music brainz id which is 60.69% of groups


,group,groupLabel,genre,groupType,inception,countryOfOrigin,recordLabel,genreLabel,groupTypeLabel,countryOfOriginLabel,recordLabelLabel,influencedby,influencedbyLabel,musicBrainzID,article,statementCount,__index_level_0__


,group,groupLabel,musicBrainzID
590,Q1165506,Ike & Tina Turner,1c7b3786-b99a-40ab-81e4-f3056da19e13
1099,Q7762513,The Scholars,29070c68-d87f-44d8-8447-753519cda4b6
1206,Q14524548,Seventeen,4bf4b341-01e9-4905-a321-1024b08bb8fb
1489,Q208767,The Stooges,794c6bf2-3241-416f-9b8f-24e2d84a1c4b
2117,Q14524548,Seventeen,b4ddad6f-2e02-4c9a-a392-79e389be95f1
...,...,...,...
214143,Q3005564,Zeds Dead,5c06eb59-2b2e-4a17-8742-30c2ed26805a
214236,Q116742236,NaN,545ac207-a7a4-4b01-bde0-3d161bc38df5
214244,Q7617074,Still Life,d12ce592-d228-44b0-917d-a3e114c4bc4e
214261,Q2712705,The Tams,a46b081a-99db-4a8e-87f1-9cd57642dfee


1242
There are 73832 in the musicBrainz dataset that are also on wikidata based on ids from wikidata


Look for missing genres


In [168]:
#What are the missing genres

genre_with_mbID = pd.read_csv("./Data/query 26 - get genres with musicbrainzID.csv")
genreDF = pd.read_csv("./Data/Query 121 - Genres and their parent genres and inception date.csv")
mbGenres = pd.read_csv("./Data/musicBrainz_Genres.csv")


In [169]:
def replaceToNulls(df, column):
    columnLabel = column + "Label"
    noLabel = df[df[column].isin(df[columnLabel])][columnLabel]
    noLabel = noLabel.to_list()
    noNameDict = {k: None for k in noLabel}
    df[columnLabel] = df[columnLabel].replace(noNameDict)

    df[column] = df[column].where(df[column].str.startswith("Q"), None)
    return df

In [170]:
#Clean the tables first
genre_with_mbID.genre = genre_with_mbID.genre.map(getEntityIDValue)
genre_with_mbID = replaceToNulls(genre_with_mbID, "genre")
print(len(genre_with_mbID["musicBrainzID"].dropna().drop_duplicates()))

#Something has more than one musicBrainz ID
genreID = genre_with_mbID[["genre","genreLabel","musicBrainzID"]].dropna().drop_duplicates()
display(genreID[genreID.duplicated(subset=["genre"])])
genre_with_mbID["genre"].unique()

2100


,genre,genreLabel,musicBrainzID
488,Q11787389,club/dance music,e5bba957-8c91-496a-a675-c6d0c6b51c33
724,Q1309336,classic female blues,c5ae8349-f729-4271-a9a5-9bf77c94b28b
809,Q5967604,urban contemporary gospel,8f40320c-f79b-4c17-a250-1f9fb550c93c
1423,Q21684437,rumba,3e67417c-f58d-43dc-a696-9b889b24641a


<ArrowStringArray>
[  'Q1092837', 'Q108716407', 'Q108807661', 'Q109930142', 'Q109941986',
 'Q108831582',  'Q10836741', 'Q108845022',   'Q1093232', 'Q108857881',
 ...
   'Q5219995',   'Q5276535',   'Q5295546',   'Q5295946',   'Q4841596',
   'Q4942839',    'Q502448',    'Q489913',   'Q4993367',  'Q47781397']
Length: 2096, dtype: str

In [171]:
genreDF.head()
genreDF[["genre","parentGenre"]] = genreDF[["genre","parentGenre"]].map(getEntityIDValue)
genreDF = replaceToNulls(genreDF, "parentGenre")
genreDF = replaceToNulls(genreDF, "genre")

wikiGenres = genreDF.merge(genre_with_mbID[["genre", "musicBrainzID"]], on="genre", how="left")
#lowerede the string titles
wikiGenres["genreLabel"] = wikiGenres["genreLabel"].str.lower()

In [172]:
wikiGenres.head()

,genre,genreLabel,parentGenre,parentGenreLabel,inception,musicBrainzID
0,Q623824,arabesque,Q1074080,music of Turkey,NaN,05b3c77c-3f55-4efc-89c7-0932f1f3f0e3
1,Q623904,andalusi classical music,Q2742742,music of Andalusia,1000-01-01T00:00:00Z,22ccf722-6f47-479c-a96f-2bc403ab98d7
2,Q623904,andalusi classical music,Q11400811,music of North Africa,1000-01-01T00:00:00Z,22ccf722-6f47-479c-a96f-2bc403ab98d7
3,Q623904,andalusi classical music,Q107189715,Arabic classical music,1000-01-01T00:00:00Z,22ccf722-6f47-479c-a96f-2bc403ab98d7
4,Q624044,arabic music,Q1503406,music of the Middle East,NaN,NaN


In [173]:
mbGenres.genre = mbGenres.genre.str.lower()
mbGenres.head()
mbGenres = mbGenres.drop_duplicates()
len(mbGenres)

2184

In [174]:
#What genres from Wikidat are in musicbrainz by id
#Use set intersection
setOfWikiMusicBrainzIDs = set(wikiGenres["musicBrainzID"].dropna().drop_duplicates())
print(len(setOfWikiMusicBrainzIDs))
setOfMBMusicBrainzIDs = set(mbGenres["musicBrainzID"].dropna().drop_duplicates())

intersection = setOfWikiMusicBrainzIDs.intersection(setOfMBMusicBrainzIDs)
genresNotOnWikidataBYID = setOfMBMusicBrainzIDs - setOfWikiMusicBrainzIDs
print(f"There {len(intersection)} genres both on MB and on wikidata")
print(f"There {len(genresNotOnWikidataBYID)} musicBrainz genres not on wikidata")

genreNamesNotOnWikidataByID = set(mbGenres[mbGenres["musicBrainzID"].isin(genresNotOnWikidataBYID)]["genre"])
print(genreNamesNotOnWikidataByID)



2100
There 2100 genres both on MB and on wikidata
There 84 musicBrainz genres not on wikidata
{'chinese revolutionary opera', 'sichuan opera', 'asmr', 'audio drama', 'opera buffa', 'opera semiseria', 'marching band', 'spoken word', 'punk poetry', 'singspiel', 'monodrama', 'yangzhou opera', 'fairy tale', 'speech', 'sticheron', 'género grande', 'sermon', 'grand opera', 'rock urbano', 'rumba flamenca', 'zarzuela', 'kunqu', 'gospel hymn', 'zeitoper', 'rustic stomp', 'valaam chant', "kalon'ny fahiny", 'opéra comique', 'verismo', 'brass band', 'budget rock', 'standup comedy', 'henan opera', 'mgodro', 'concert band', 'battle record', 'interview', 'beijing opera', 'whale song', 'guayla', 'lecture', 'tassa', 'jungle techno', 'sketch comedy', 'sevillanas', 'animal sounds', 'cumbia sureña peruana', 'cumbia norteña peruana', 'romantische oper', 'operetta', 'audio documentary', 'partyschlager', 'broadband noise', 'korean revolutionary opera', 'chakacha', 'prank calls', 'nature sounds', 'poetry', 'o

In [175]:
wikiGenres[wikiGenres["genre"] == "Q101002867"]

,genre,genreLabel,parentGenre,parentGenreLabel,inception,musicBrainzID
8510,Q101002867,mgodro,NaN,None,NaN,NaN


In [176]:
#Are these names in wikiGenres

genresNotAttachedToTheirID = wikiGenres[wikiGenres["genreLabel"].isin(genreNamesNotOnWikidataByID)][["genre", "genreLabel"]].dropna().drop_duplicates()
print(f"These genres have names matching the names in the music brainz genres. There {len(genresNotAttachedToTheirID["genre"])} genres.")
genresNotAttachedToTheirID.to_csv("./missingValuesStore/genresInMBAndWikidataButNoMbID.csv")
genresNotPresentOnWikidata = genreNamesNotOnWikidataByID - set(genresNotAttachedToTheirID["genreLabel"])

print(f"There are {len(genresNotPresentOnWikidata)} not present on wikidata.")
print(mbGenres[mbGenres["genre"].isin(genresNotPresentOnWikidata)])



These genres have names matching the names in the music brainz genres. There 9 genres.
There are 75 not present on wikidata.
                             musicBrainzID              genre
78    6ce4f333-ee9b-4da9-a044-875ddae2fb38      animal sounds
101   19191e39-9867-45bf-9347-dbfcaf9a3f72               asmr
106   894cbdf2-c1b0-4b65-8ab6-d828a76c5d34  audio documentary
107   cc6b682e-5204-4f8a-a620-b63ccdf23d9f        audio drama
135   cd8c060f-15a5-4442-a7e7-f2bb10a7d28d       ballad opera
...                                    ...                ...
2151  9b8aa719-34c6-40da-8d1c-64dfdd20592a     yangzhou opera
2158  f8b8e9f5-f475-46ac-95b4-8e9a889e6a1c          yue opera
2164  0d82a4bc-f775-48ae-a5cc-73ff2aa5510f           zarzuela
2165  d85ac10f-deb1-410c-af5c-4d40f77003d9   zarzuela barroca
2166  642639a9-c95a-48ec-9695-59b7a5935e67           zeitoper

[75 rows x 2 columns]
